# 07 · External validation — do the HAM10000 models generalise?

A HAM10000-only study never shows the model works **outside** HAM10000. This
notebook scores the **already-trained** models (no retraining) on a *different*
skin-lesion dataset — by default **PAD-UFES-20** — and reports the same
medical-grade metrics as notebook 06, so we get an honest out-of-distribution
estimate and a per-stage **generalisation gap**.

PAD-UFES-20 is a small (2,298-image), clinically-annotated dataset of
**smartphone photos** collected in Brazil — chosen here instead of ISIC 2019
because it's a fraction of the size (no multi-GB download) while still
overlapping with our label set. Note this is a **modality shift as well as a
dataset shift**: PAD-UFES-20 images are ordinary clinical photos, not
dermoscopy, so a performance drop here reflects both.

Two correctness traps are handled explicitly (see `src/external.py`):

1. **Leakage.** PAD-UFES-20 is collected independently from ISIC/HAM10000 (own
   patients, own smartphone captures), so overlap with the HAM training
   manifest (`models/ham_split.csv`) should be **zero**. We still run the
   same exact-ID overlap check as a safety net — it's a no-op here, but keeps
   this notebook identical in structure if you later swap in an ISIC-derived
   external set.
2. **Label / subclass mismatch.** PAD-UFES-20 has six diagnoses: `BCC`, `SCC`,
   `ACK`, `SEK`, `MEL`, `NEV`. We map these onto our `dx` codes and **keep only
   the subclasses in our own hierarchy**:
   - Benign (Stage 2a): `NEV→nv`, `SEK→bkl`. (`vasc`, `df` do not exist in
     this dataset — those rows just have zero support at eval time, same as
     any missing class.)
   - Malignant (Stage 2b): `MEL→mel`, `BCC→bcc`, `ACK→akiec`.
   - `SCC` is **dropped** — it isn't one of the seven subclasses in our
     hierarchy (`nv, bkl, vasc, df, mel, bcc, akiec`), so keeping it would
     silently inflate/deflate Stage 1 malignant counts with a class our
     Stage 2b head was never trained to predict.
   PAD-UFES-20's `region` field is crosswalked to HAM `localization` values;
   anything unmapped falls to the `unknown` bucket, same convention as before.

---
*Part of: **A Multimodal, Explainable Hierarchical Framework for Skin-Lesion
Classification with Diffusion-Based Augmentation of Rare Malignancies.***

In [1]:
# --- Dependencies (Colab already has TF; this pins the extras) ----------------
!pip -q install scikit-learn matplotlib pillow 
print('deps ready')

deps ready



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# --- Point this at your project folder (same convention as notebook 06) -------
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Skin_Cancer_Research_Grant2024'
except Exception:
    PROJECT_DIR = os.path.abspath('../')

SRC        = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
DATA_DIR   = os.path.join(PROJECT_DIR, 'data')
FIG_DIR    = os.path.join(PROJECT_DIR, 'report', 'figures')
SPLIT_CSV  = os.path.join(MODELS_DIR, 'ham_split.csv')
PREP_JSON  = os.path.join(MODELS_DIR, 'preprocessing.json')
for d in (MODELS_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.isdir(SRC), f'src/ not found at {SRC} — set PROJECT_DIR correctly'
sys.path.insert(0, SRC)

# --- External dataset config --------------------------------------------------
# Put the PAD-UFES-20 release under DATA_DIR/pad_ufes_20/ so these resolve:
PAD_DIR      = os.path.join(DATA_DIR, 'pad_ufes_20')
PAD_CSV      = os.path.join(PAD_DIR, 'metadata.csv')
PAD_IMAGES   = os.path.join(PAD_DIR, 'images')   # see loader below — also checks imgs_part_1/2/3
DATASET_NAME = 'PAD-UFES-20 (clinical images, filtered to our 7 subclasses)'

# Only these subclasses are kept — everything else (e.g. PAD's SCC) is dropped.
KEEP_DX = {'nv', 'bkl', 'vasc', 'mel', 'bcc', 'akiec'}

MAX_E2E      = 3000   # cap for the per-image end-to-end loop; None = all images
print('PROJECT_DIR =', PROJECT_DIR)

PROJECT_DIR = C:\Users\samjh\Downloads\Skin_Cancer_Research_Grant2024


## Get the external dataset

**PAD-UFES-20 (default).** Much smaller than ISIC 2019 (2,298 images total).
Download it from [Mendeley Data](https://data.mendeley.com/datasets/zr7vgbcyr2/1)
(or a Kaggle mirror) and place it under `DATA_DIR/pad_ufes_20/` as:

```
pad_ufes_20/
├─ metadata.csv                # one row per image: img_id, diagnostic, age, region, ...
└─ images/                     # <img_id>.png files (or imgs_part_1/2/3 subfolders — handled below)
```

Example Kaggle route (needs `kaggle.json` uploaded to the session):

```python
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mahdavi1202/skin-cancer -p {PAD_DIR} --unzip
```

The exact mirror/slug does not matter — only that `metadata.csv` and the image
files resolve under `PAD_DIR`. If your mirror keeps the original
`imgs_part_1/`, `imgs_part_2/`, `imgs_part_3/` split instead of one flat
`images/` folder, that's fine — the loader below searches all of them.

**Note on subclasses.** PAD-UFES-20 only has six diagnoses (`BCC, SCC, ACK,
SEK, MEL, NEV`) — there's no `vasc` or `df` here. The loader keeps only the
five that map onto our hierarchy (`NEV→nv`, `SEK→bkl`, `MEL→mel`,
`BCC→bcc`, `ACK→akiec`) and drops `SCC` rows entirely, per `KEEP_DX` above.

In [9]:
# --- Load PAD-UFES-20 into a HAM-shaped dataframe ------------------------------
import pandas as pd
import external as X

assert os.path.isdir(PAD_DIR), f'missing: {PAD_DIR} — see the cell above'

def find_pad_metadata_csv(pad_dir, expected=None):
    """Auto-locate the metadata CSV instead of assuming a fixed name/path —
    handles both metadata.csv sitting directly in pad_dir and a metadata/
    subfolder containing it."""
    if expected and os.path.exists(expected):
        return expected
    candidates = []
    for root, _dirs, files in os.walk(pad_dir):
        for f in files:
            if f.lower().endswith('.csv'):
                candidates.append(os.path.join(root, f))
    if not candidates:
        raise FileNotFoundError(f'No .csv file found anywhere under {pad_dir}')
    # prefer a file with 'metadata' in the name if there's more than one
    for c in candidates:
        if 'metadata' in os.path.basename(c).lower():
            return c
    return candidates[0]

PAD_CSV = find_pad_metadata_csv(PAD_DIR, expected=PAD_CSV)
print('using metadata csv:', PAD_CSV)

# PAD-UFES-20 diagnostic code -> our dx code. SCC has no entry on purpose:
# it isn't one of our seven trained subclasses, so those rows get dropped.
PAD_DX_MAP = {
    'NEV': 'nv',
    'SEK': 'bkl',
    'MEL': 'mel',
    'BCC': 'bcc',
    'ACK': 'akiec',
}
PAD_MALIGNANT = {'mel', 'bcc', 'akiec'}

# HAM10000 `localization` vocabulary, crosswalked from PAD's `region` field.
PAD_LOCALIZATION_MAP = {
    'face': 'face', 'scalp': 'scalp', 'ear': 'ear', 'neck': 'neck',
    'chest': 'chest', 'back': 'back', 'abdomen': 'abdomen',
    'arm': 'upper extremity', 'forearm': 'upper extremity', 'hand': 'hand',
    'thigh': 'lower extremity', 'foot': 'foot', 'nose': 'face', 'lip': 'face',
}

def build_pad_image_index(pad_dir):
    """Walk the whole PAD_DIR tree once and map filename -> full path.
    Handles any folder layout (flat images/, imgs_part_1/2/3, or the
    doubly-nested imgs_part_1/imgs_part_1/ that some mirrors produce)
    instead of guessing candidate paths."""
    index = {}
    for root, _dirs, files in os.walk(pad_dir):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                index.setdefault(f, os.path.join(root, f))
    return index

def load_pad_ufes20_dataframe(csv_path, pad_dir, keep_dx):
    raw = pd.read_csv(csv_path)
    df = pd.DataFrame({
        'image_id': raw['img_id'].astype(str).str.replace(r'\.(png|jpg|jpeg)$', '', regex=True),
        'dx': raw['diagnostic'].map(PAD_DX_MAP),
        'age': raw.get('age'),
        'sex': raw.get('gender'),
        'localization': raw.get('region', pd.Series(dtype=str)).astype(str).str.lower()
                            .map(PAD_LOCALIZATION_MAP).fillna('unknown'),
    })

    image_index = build_pad_image_index(pad_dir)
    print(f'indexed {len(image_index)} image files under {pad_dir}')
    df['image_path'] = raw['img_id'].astype(str).map(image_index)

    missing = df['image_path'].isna()
    if missing.any():
        print(f'WARNING: {missing.sum()} rows have no matching image file — dropping them')
        print('  example missing ids:', df.loc[missing, 'image_id'].head(5).tolist())
        df = df[~missing].reset_index(drop=True)

    # Drop anything that isn't NEV/SEK/MEL/BCC/ACK (e.g. SCC), then enforce
    # the exact subclass allow-list requested for this study.
    df = df.dropna(subset=['dx'])
    df = df[df['dx'].isin(keep_dx)].reset_index(drop=True)

    df['binary_label'] = df['dx'].apply(lambda d: 'malignant' if d in PAD_MALIGNANT else 'benign')
    return df

ext_df = load_pad_ufes20_dataframe(PAD_CSV, PAD_DIR, KEEP_DX)
n_raw = len(ext_df)
print(f'loaded {n_raw} usable PAD-UFES-20 rows (filtered to {sorted(KEEP_DX)})')
print(ext_df['dx'].value_counts())

using metadata csv: C:\Users\samjh\Downloads\Skin_Cancer_Research_Grant2024\data\pad_ufes_20\metadata.csv
indexed 2298 image files under C:\Users\samjh\Downloads\Skin_Cancer_Research_Grant2024\data\pad_ufes_20
loaded 2106 usable PAD-UFES-20 rows (filtered to ['akiec', 'bcc', 'bkl', 'mel', 'nv', 'vasc'])
dx
bcc      845
akiec    730
nv       244
bkl      235
mel       52
Name: count, dtype: int64


In [11]:
# --- Leakage guard: drop any images that were in HAM training -----------------

HAM_META = os.path.join(DATA_DIR, 'HAM10000_metadata.csv') 
train_ids = X.load_training_image_ids(SPLIT_CSV, HAM_META)
ext_df, n_removed = X.remove_training_overlap(ext_df, train_ids)
print(f'removed {n_removed} HAM-overlap images; {len(ext_df)} remain for external eval')
if n_removed == 0:
    print('  (0 removed — expected: PAD-UFES-20 is disjoint from HAM/ISIC)')

removed 0 HAM-overlap images; 2106 remain for external eval
  (0 removed — expected: PAD-UFES-20 is disjoint from HAM/ISIC)


## Run the external validation

Same protocol as notebook 06 (baseline vs. multimodal per stage, McNemar, ECE,
confusion/ROC/reliability figures, end-to-end pipeline) — but every image is
unseen, from a different source (and a different modality: clinical photos
rather than dermoscopy). Figures are written as `report/figures/ext_*.png`;
metrics to `report/external_results.json`.

In [12]:
import json
import metadata as M
import evaluate as ev

proc = M.MetadataProcessor.load(PREP_JSON)

external = X.run_external_validation(
    ext_df, MODELS_DIR, proc, FIG_DIR,
    dataset_name=DATASET_NAME,
    n_overlap_removed=n_removed,
    n_total=n_raw,
    max_e2e=MAX_E2E,
)

ev.save_metrics_json(external, os.path.join(PROJECT_DIR, 'report', 'external_results.json'))
print(json.dumps({k: v for k, v in external.items() if k in ('dataset', 'end_to_end')}, indent=2))

{
  "end_to_end": {
    "stage1_accuracy": 0.5992402659069326,
    "malignant_sensitivity": 0.544560540872772,
    "end_to_end_subtype_accuracy": 0.22317188983855651,
    "n_test": 2106,
    "n_available": 2106,
    "capped": false
  },
  "dataset": {
    "name": "PAD-UFES-20 (clinical images, filtered to our 7 subclasses)",
    "n_total": 2106,
    "n_overlap_removed": 0,
    "n_evaluated": 2106,
    "class_counts": {
      "bcc": 845,
      "akiec": 730,
      "nv": 244,
      "bkl": 235,
      "mel": 52
    }
  }
}


In [13]:
# --- Readable summary table ---------------------------------------------------
def _fmt(x):
    return '—' if x is None else f'{x:.3f}'

print('External validation —', external['dataset']['name'])
print(f"  evaluated {external['dataset']['n_evaluated']} images "
      f"({external['dataset']['n_overlap_removed']} HAM-overlap removed)\n")
for stage, pretty in [('stage1','Stage 1'),('benign','Stage 2a'),('malignant','Stage 2b')]:
    R = external.get(stage)
    if not R:
        continue
    print(f'  [{pretty}]  n={R["n"]}')
    for mdl in ('baseline','multimodal'):
        m = R.get(mdl)
        if not m:
            continue
        auc = m.get('roc_auc', m.get('roc_auc_ovr_macro'))
        extra = ''
        if 'sensitivity' in m:
            extra = f"  sens={_fmt(m['sensitivity'])}  spec={_fmt(m['specificity'])}"
        print(f'    {mdl:<11} acc={_fmt(m["accuracy"])}  macroF1={_fmt(m["macro_f1"])}  AUC={_fmt(auc)}{extra}')
    if 'mcnemar' in R:
        print(f"    McNemar exact p = {R['mcnemar'].get('p_value_exact')}")
    print()

External validation — PAD-UFES-20 (clinical images, filtered to our 7 subclasses)
  evaluated 2106 images (0 HAM-overlap removed)

  [Stage 1]  n=2106
    baseline    acc=0.491  macroF1=0.480  AUC=0.622  sens=0.409  spec=0.768
    multimodal  acc=0.532  macroF1=0.519  AUC=0.703  sens=0.448  spec=0.816
    McNemar exact p = 0.0001740243802692251

  [Stage 2a]  n=479
    baseline    acc=0.503  macroF1=0.370  AUC=—
    multimodal  acc=0.509  macroF1=0.362  AUC=—
    McNemar exact p = 0.8653735109027345

  [Stage 2b]  n=1627
    baseline    acc=0.403  macroF1=0.320  AUC=0.648
    multimodal  acc=0.254  macroF1=0.232  AUC=0.623
    McNemar exact p = 1.5325964276916215e-26



## Generalisation gap: internal (HAM test) vs. external

The headline of an external-validation study is the **gap** between held-out
HAM10000 performance (notebook 06 → `report/results.json`) and this external set.
A small gap = the model generalises; a large drop = HAM-specific shortcuts.

In [14]:
import json
res_path = os.path.join(PROJECT_DIR, 'report', 'results.json')
if os.path.exists(res_path):
    internal = json.load(open(res_path))
    def s1(d, k):
        return d.get('stage1', {}).get('multimodal', {}).get(k)
    rows = [('Sensitivity','sensitivity'),('Specificity','specificity'),
            ('ROC-AUC','roc_auc'),('Macro-F1','macro_f1'),('Accuracy','accuracy')]
    print(f"{'Stage-1 (multimodal)':<22}{'internal':>10}{'external':>10}{'gap':>9}")
    for name, key in rows:
        i, e = s1(internal, key), s1(external, key)
        if i is None or e is None:
            continue
        print(f'{name:<22}{i:>10.3f}{e:>10.3f}{e-i:>+9.3f}')
else:
    print('run notebook 06 first to produce report/results.json for the comparison')

Stage-1 (multimodal)    internal  external      gap
Sensitivity                0.536     0.448   -0.088
Specificity                0.898     0.816   -0.081
ROC-AUC                    0.852     0.703   -0.149
Macro-F1                   0.720     0.519   -0.201
Accuracy                   0.827     0.532   -0.295
